# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: K-means** </center>
---
**Profesor**: Pablo Camarillo Ramirez
---
***Alumno***: Nicolas Navarro Valenzuela

# Create SparkSession

In [1]:
from spark_utils import SparkUtils
su = SparkUtils("ML: K-means", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 00:34:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Clustering with 2D points

In [2]:
# Sample data in Python (e.g., 2D points)
data = [
    (0, 1.0, 1.0),
    (1, 2.0, 1.0),
    (2, 4.0, 5.0),
    (3, 5.0, 5.0),
    (4, 10.0, 10.0),
    (5, 12.0, 11.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("id", "int"), ("x", "float"), ("y", "float")])

# Create DataFrame for k means
random_points_df = su.spark.createDataFrame(data, schema)

## Assemble the features into a single vector column

In [3]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")
assembled_df = assembler.transform(random_points_df)

## Configure K-means

In [19]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans().setK(3).setSeed(73)

## Train model

In [20]:
model = kmeans.fit(assembled_df)
print("K-means model trained successfully")
kmeans_model_path = "/opt/spark/work-dir/data/mlmodels/kmeans/2D"
model.write().overwrite().save(kmeans_model_path)
model.__class__

K-means model trained successfully


pyspark.ml.clustering.KMeansModel

## Get Predictions

In [21]:
from pyspark.ml.clustering import KMeansModel
k_model = KMeansModel.load(kmeans_model_path)
predictions = k_model.transform(assembled_df)

In [22]:
predictions.show()

+---+----+----+-----------+----------+
| id|   x|   y|   features|prediction|
+---+----+----+-----------+----------+
|  0| 1.0| 1.0|  [1.0,1.0]|         2|
|  1| 2.0| 1.0|  [2.0,1.0]|         2|
|  2| 4.0| 5.0|  [4.0,5.0]|         0|
|  3| 5.0| 5.0|  [5.0,5.0]|         0|
|  4|10.0|10.0|[10.0,10.0]|         1|
|  5|12.0|11.0|[12.0,11.0]|         1|
+---+----+----+-----------+----------+



## Evaluate model

In [23]:
from pyspark.ml.evaluation import ClusteringEvaluator

# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette}")

# Show the result
print("Cluster Centers: ")
for center in model.clusterCenters():
    print(center)

Silhouette score: 0.9494652547284126
Cluster Centers: 
[4.5 5. ]
[11.  10.5]
[1.5 1. ]


# Lab 13: Clustering Wine dataset with K-means

In [25]:
# Downlod dataset from https://www.kaggle.com/datasets/harrywang/wine-dataset-for-clustering

columns_types = [("Alcohol", "float"),
                                     ("Malic_Acid", "float"),
                                     ("Ash", "float"),
                                     ("Ash_Alcanity", "float"),
                                     ("Magnesium", "float"),
                                     ("Total_Phenols", "float"),
                                     ("Flavanoids", "float"),
                                     ("Nonflavanoid_Phenols", "float"),
                                     ("Proanthocyanins", "float"),
                                     ("Color_Intensity", "float"),
                                     ("Hue", "float"),
                                     ("OD280", "float"),
                                     ("Proline", "float")]

# Define schema for the DataFrame
wines_schema = SparkUtils.generate_schema(columns_types)

# Create DataFrame from wines csv
wines_df = su.spark \
                    .read \
                    .option("header", "true") \
                    .schema(wines_schema) \
                    .csv("/opt/spark/work-dir/data/ml/kmeans")


assembler = VectorAssembler(inputCols=[x for x,_ in columns_types], outputCol="features")
assembled_df = assembler.transform(wines_df)


In [40]:

# TODO: Find the optimal K
# TODO: Add the code here to iterate from k = 2, 4, .., 10 and get the silhouette score for each k

silhouette_scores = {}

# Iteramos sobre diferentes valores de K
for k in range(2, 11, 2):  # k = 2, 4, 6, 8, 10
    print(f"\nTraining model with k = {k} clusters...")
    
    # Crear modelo K-means con el valor actual de k
    kmeans = KMeans().setK(k).setSeed(13)
    
    # Entrenar el modelo
    model = kmeans.fit(assembled_df)
    
    # Hacer predicciones (asignar cada vino a un cluster)
    predictions = model.transform(assembled_df)
    
    # Calcular el Silhouette Score (métrica de calidad del clustering)
    evaluator = ClusteringEvaluator()
    silhouette = evaluator.evaluate(predictions)
    
    # Guardar el resultado
    silhouette_scores[k] = silhouette
    print(f"Silhouette Score: {silhouette:.4f}")

print("\n")
for k in sorted(silhouette_scores.keys()):
    score = silhouette_scores[k]
    print(f"for K = {k:2d} Silhouette Score = {score:.4f}")

# Encontrar el K óptimo (con el mejor score)
optimal_k = max(silhouette_scores, key=silhouette_scores.get)
best_score = silhouette_scores[optimal_k]

print("\n" + "=" * 60)
print(f"Optimal K found: K = {optimal_k}")
print("=" * 60)

# Entrenar el modelo final con el K óptimo
print(f"\nTraining final model with K = {optimal_k}...")
k = optimal_k
kmeans = KMeans().setK(k).setSeed(13)
model = kmeans.fit(assembled_df)

print(f"Final model trained successfully with {k} clusters")



Training model with k = 2 clusters...
Silhouette Score: 0.8194

Training model with k = 4 clusters...
Silhouette Score: 0.7303

Training model with k = 6 clusters...
Silhouette Score: 0.7359

Training model with k = 8 clusters...
Silhouette Score: 0.6987

Training model with k = 10 clusters...
Silhouette Score: 0.6915


for K =  2 Silhouette Score = 0.8194
for K =  4 Silhouette Score = 0.7303
for K =  6 Silhouette Score = 0.7359
for K =  8 Silhouette Score = 0.6987
for K = 10 Silhouette Score = 0.6915

Optimal K found: K = 2

Training final model with K = 2...
Final model trained successfully with 2 clusters


26/04/28 01:14:21 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-252fbc94-b5b1-4473-b3ef-7c7d0d09ad02. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-252fbc94-b5b1-4473-b3ef-7c7d0d09ad02
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:372)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:368)
	at scala.collection.ArrayOps$.foreach$

In [ ]:
su.spark.stop()